# 01 - Análise exploratória: entregas de moto no ABC paulista e em Mauá

**Objetivo:** conhecer a base simulada de entregas antes de qualquer limpeza ou modelagem.

**Perguntas de negócio que guiam esta etapa**
1. Qual o volume de entregas e como ele se distribui por status?
2. Distância, tempo de trânsito e valor estão em faixas plausíveis para moto em área urbana?
3. O horário de pico de trânsito muda tempo, atraso e cancelamento?
4. Quais cidades e bairros concentram a coleta e a entrega?
5. Entregas entre cidades se comportam diferente das entregas dentro da mesma cidade?

**Fonte:** `data/raw/dados_entregas_simulados.csv`, gerado por `src/logistica_rotas/ingestao/simulacao.py` (`random_state=42`).

**Real x simulado:** municípios (Santo André, São Bernardo do Campo, São Caetano do Sul, Diadema e Mauá), bairros e vias são reais. A posição dos bairros, os pedidos, tempos e valores são simulados. Detalhes em `docs/dicionario_dados.md`.

> Por que simular? Permite controlar as regras de negócio (prazo, velocidade, pico) e saber de antemão o que a análise deveria encontrar - bom para treinar e validar o método.

## 1. Configuração

Localiza a raiz do projeto e coloca `src/` no caminho de importação, para usar as funções do pacote `logistica_rotas`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Sobe pastas até achar o CLAUDE.md (raiz do projeto), funcione o notebook de onde for aberto.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists())
sys.path.insert(0, str(RAIZ / "src"))

from logistica_rotas.ingestao.leitura import carregar_entregas, rotulo_local

pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)
print("Raiz do projeto:", RAIZ)

Raiz do projeto: C:\Users\Usuario\OneDrive\Documentos\projeto_logistica_rotas


## 2. Fonte de dados

`carregar_entregas()` lê o CSV com os tipos corretos (data com fuso `America/Sao_Paulo`, textos repetidos como categoria). Se o CSV não existir (ele não vai para o Git), é gerado na hora - mesma seed, mesmos dados.

**Atenção:** o bairro "Centro" existe nas 5 cidades. Por isso criamos `local_origem` / `local_destino` ("Centro · SBC"), para nunca somar bairros homônimos de cidades diferentes.

In [2]:
df = carregar_entregas()
df["local_origem"] = rotulo_local(df["bairro_origem"], df["cidade_origem"])
df["local_destino"] = rotulo_local(df["bairro_destino"], df["cidade_destino"])
df["trajeto"] = np.where(df["cidade_origem"].astype(str) == df["cidade_destino"].astype(str),
                         "mesma cidade", "entre cidades")

print(f"{df.shape[0]:,} linhas x {df.shape[1]} colunas".replace(",", "."))
df.head()

5.000 linhas x 16 colunas


,id_pedido,data_hora_pedido,cidade_origem,bairro_origem,cidade_destino,bairro_destino,endereco_destino,distancia_km,horario_pico,tempo_transito_min,prazo_prometido_min,valor_entrega_brl,status,local_origem,local_destino,trajeto
0,PED000001,2026-09-01 10:06:00-03:00,Diadema,Centro,Diadema,Eldorado,"Rua Manoel da Nóbrega, 738",2.84,nao,10.00,20,10.25,entregue,Centro · Diadema,Eldorado · Diadema,mesma cidade
1,PED000002,2026-09-01 10:21:00-03:00,Diadema,Centro,Diadema,Conceição,"Avenida Piraporinha, 2968",2.88,nao,17.20,20,10.31,entregue,Centro · Diadema,Conceição · Diadema,mesma cidade
2,PED000003,2026-09-01 10:32:00-03:00,Santo André,Parque das Nações,Santo André,Utinga,"Avenida Queirós dos Santos, 2911",7.55,nao,30.60,35,17.32,entregue,Parque das Nações · SA,Utinga · SA,mesma cidade
3,PED000004,2026-09-01 11:00:00-03:00,Santo André,Centro,Diadema,Eldorado,"Avenida Piraporinha, 39",12.56,nao,28.20,45,24.83,entregue,Centro · SA,Eldorado · Diadema,entre cidades
4,PED000005,2026-09-01 11:05:00-03:00,São Bernardo do Campo,Centro,São Bernardo do Campo,Ferrazópolis,"Rua Jurubatuba, 1205",3.48,nao,12.20,25,11.21,entregue,Centro · SBC,Ferrazópolis · SBC,mesma cidade


## 3. Estrutura e tipos

Confere se cada coluna foi lida com o tipo certo. Unidades estão no nome: `_km`, `_min`, `_brl`.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype                            
---  ------               --------------  -----                            
 0   id_pedido            5000 non-null   str                              
 1   data_hora_pedido     5000 non-null   datetime64[us, America/Sao_Paulo]
 2   cidade_origem        5000 non-null   category                         
 3   bairro_origem        5000 non-null   category                         
 4   cidade_destino       5000 non-null   category                         
 5   bairro_destino       5000 non-null   category                         
 6   endereco_destino     5000 non-null   str                              
 7   distancia_km         5000 non-null   float64                          
 8   horario_pico         5000 non-null   category                         
 9   tempo_transito_min   4790 non-null   float64                   

## 4. Checklist de validação

Itens do checklist da skill `logistica-dados`. Aqui só **identificamos** problemas; o tratamento fica para o notebook de limpeza.

In [4]:
concluidas = df["status"] != "cancelado"
velocidade_kmh = df["distancia_km"] / (df["tempo_transito_min"] / 60)

checagens = pd.Series({
    "id_pedido duplicado": df["id_pedido"].duplicated().sum(),
    "nulos em tempo_transito_min (total)": df["tempo_transito_min"].isna().sum(),
    "nulos em tempo_transito_min (cancelados)": df.loc[~concluidas, "tempo_transito_min"].isna().sum(),
    "nulos em tempo_transito_min (não cancelados)": df.loc[concluidas, "tempo_transito_min"].isna().sum(),
    "nulos em cidade/bairro/endereço": df[["cidade_origem", "bairro_origem", "cidade_destino",
                                            "bairro_destino", "endereco_destino"]].isna().sum().sum(),
    "distância <= 0": (df["distancia_km"] <= 0).sum(),
    "tempo <= 0": (df["tempo_transito_min"] <= 0).sum(),
    "velocidade média < 3 km/h (suspeito)": (velocidade_kmh < 3).sum(),
    "velocidade média > 60 km/h (impossível p/ moto urbana)": (velocidade_kmh > 60).sum(),
}, name="quantidade")
checagens.to_frame()

,quantidade
id_pedido duplicado,0
nulos em tempo_transito_min (total),210
nulos em tempo_transito_min (cancelados),210
nulos em tempo_transito_min (não cancelados),0
nulos em cidade/bairro/endereço,0
distância <= 0,0
tempo <= 0,0
velocidade média < 3 km/h (suspeito),53
velocidade média > 60 km/h (impossível p/ moto urbana),0


**Leitura:** os nulos de tempo devem coincidir com os cancelados (entrega não aconteceu). Velocidades médias muito baixas indicam erro de registro do tempo - candidatos a outlier.

## 5. Estatísticas descritivas - variáveis numéricas

In [5]:
numericas = ["distancia_km", "tempo_transito_min", "prazo_prometido_min", "valor_entrega_brl"]
desc = df[numericas].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).T
desc["cv_%"] = desc["std"] / desc["mean"] * 100  # coeficiente de variação
desc

,count,mean,std,min,25%,50%,75%,95%,99%,max,cv_%
distancia_km,5000.00,4.90,4.11,0.50,2.18,3.71,6.27,13.58,20.46,27.42,83.75
tempo_transito_min,4790.00,19.23,15.73,3.30,11.10,15.30,22.30,43.00,71.88,371.80,81.82
prazo_prometido_min,5000.00,26.69,10.47,15.00,20.00,25.00,30.00,50.00,65.00,85.00,39.23
valor_entrega_brl,5000.00,13.86,6.46,6.74,9.55,12.00,15.98,27.40,37.55,56.55,46.60


**Como ler:** compare média e mediana (50%). Média bem acima da mediana indica cauda longa à direita - o máximo do tempo de trânsito, muito acima do percentil 99, é sinal de outlier. A distância também tem cauda longa: são as entregas entre cidades.

## 6. Variáveis categóricas

In [6]:
status = pd.DataFrame({
    "qtd": df["status"].value_counts(),
    "%": df["status"].value_counts(normalize=True) * 100,
})
status

,qtd,%
status,,
entregue,4446,88.92
com_atraso,344,6.88
cancelado,210,4.20


In [7]:
pd.crosstab(df["horario_pico"], df["status"], normalize="index", margins=True) * 100

status,cancelado,com_atraso,entregue
horario_pico,,,
nao,3.88,3.85,92.27
sim,5.49,18.98,75.52
All,4.20,6.88,88.92


### Cidades

Participação de cada cidade na coleta e na entrega.

In [8]:
cidades = pd.DataFrame({
    "coletas_%": df["cidade_origem"].value_counts(normalize=True) * 100,
    "entregas_%": df["cidade_destino"].value_counts(normalize=True) * 100,
}).sort_values("coletas_%", ascending=False)
cidades

,coletas_%,entregas_%
São Bernardo do Campo,29.56,28.20
Santo André,26.96,26.38
Diadema,16.48,17.20
Mauá,15.68,16.28
São Caetano do Sul,11.32,11.94


### Fluxo entre cidades

Linhas = cidade de coleta; colunas = cidade de entrega (% da linha). A diagonal mostra as entregas dentro da mesma cidade.

In [9]:
pd.crosstab(df["cidade_origem"], df["cidade_destino"], normalize="index") * 100

cidade_destino,Diadema,Mauá,Santo André,São Bernardo do Campo,São Caetano do Sul
cidade_origem,,,,,
Diadema,80.58,4.85,5.46,6.19,2.91
Mauá,5.36,77.30,6.51,7.53,3.32
Santo André,4.60,5.04,79.75,8.23,2.37
São Bernardo do Campo,5.21,5.48,8.25,77.88,3.18
São Caetano do Sul,2.65,3.36,4.59,6.71,82.69


### Bairros (top 15)

Sempre com a cidade junto (`local_origem` / `local_destino`).

In [10]:
top_coleta = df["local_origem"].value_counts().head(15).rename("coletas")
top_entrega = df["local_destino"].value_counts().head(15).rename("entregas")
pd.concat([top_coleta.reset_index(), top_entrega.reset_index()], axis=1)

,local_origem,coletas,local_destino,entregas
0,Centro · SBC,372,Rudge Ramos · SBC,161
1,Centro · SA,318,Jardim do Mar · SBC,155
2,Centro · Diadema,261,Vila Guiomar · SA,151
3,Centro · Mauá,241,Vila Bastos · SA,148
4,Ferrazópolis · SBC,144,Vila Assunção · SA,144
5,Assunção · SBC,142,Baeta Neves · SBC,144
6,Centro · SCS,133,Assunção · SBC,144
7,Baeta Neves · SBC,124,Centro · SBC,142
8,Vila Guiomar · SA,124,Paulicéia · SBC,141
9,Vila Bastos · SA,123,Ferrazópolis · SBC,138


## 7. Primeiros indicadores

Definições (da skill `logistica-dados`; registrar em `docs/` antes de virar KPI oficial):
- **OTD** = entregas no prazo / entregas concluídas. Cancelados ficam fora do denominador.
- **Taxa de cancelamento** = cancelados / total de pedidos.
- Para tempo, usamos a **mediana**, que é menos sensível aos outliers vistos acima.

In [11]:
def resumo_operacional(grupo: pd.DataFrame) -> pd.Series:
    """Indicadores básicos de um grupo de entregas."""
    feitas = grupo[grupo["status"] != "cancelado"]
    return pd.Series({
        "pedidos": len(grupo),
        "otd_%": (feitas["status"] == "entregue").mean() * 100,
        "cancelamento_%": (grupo["status"] == "cancelado").mean() * 100,
        "distancia_media_km": grupo["distancia_km"].mean(),
        "tempo_mediano_min": feitas["tempo_transito_min"].median(),
        "valor_medio_brl": grupo["valor_entrega_brl"].mean(),
    })

geral = resumo_operacional(df).rename("geral")
por_pico = df.groupby("horario_pico", observed=True).apply(resumo_operacional, include_groups=False)
pd.concat([geral.to_frame().T, por_pico]).astype({"pedidos": int})

,pedidos,otd_%,cancelamento_%,distancia_media_km,tempo_mediano_min,valor_medio_brl
geral,5000,92.82,4.20,4.90,15.30,13.86
nao,3999,95.99,3.88,4.95,14.70,13.42
sim,1001,79.92,5.49,4.69,18.70,15.63


### Mesma cidade x entre cidades

In [12]:
df.groupby("trajeto").apply(resumo_operacional, include_groups=False).astype({"pedidos": int})

,pedidos,otd_%,cancelamento_%,distancia_media_km,tempo_mediano_min,valor_medio_brl
trajeto,,,,,,
entre cidades,1036,84.48,4.25,10.84,30.40,23.05
mesma cidade,3964,95.00,4.19,3.35,13.60,11.46


### Por cidade de destino

In [13]:
(df.groupby("cidade_destino", observed=True)
   .apply(resumo_operacional, include_groups=False)
   .astype({"pedidos": int})
   .sort_values("otd_%"))

,pedidos,otd_%,cancelamento_%,distancia_media_km,tempo_mediano_min,valor_medio_brl
cidade_destino,,,,,,
Mauá,814,91.74,3.32,6.03,16.00,15.56
Diadema,860,92.48,4.19,5.01,14.35,14.06
São Bernardo do Campo,1410,92.74,4.33,4.70,16.00,13.57
Santo André,1319,92.77,4.55,4.80,16.60,13.70
São Caetano do Sul,597,95.10,4.36,3.91,11.70,12.33


### Por hora do pedido

In [14]:
df["hora"] = df["data_hora_pedido"].dt.hour
por_hora = df.groupby("hora").apply(resumo_operacional, include_groups=False).astype({"pedidos": int})
por_hora[["pedidos", "otd_%", "tempo_mediano_min"]]

,pedidos,otd_%,tempo_mediano_min
hora,,,
8,102,85.57,18.80
9,179,97.62,14.65
10,231,94.59,16.05
11,491,95.99,14.80
12,589,95.73,15.10
13,443,96.13,14.90
14,240,96.51,14.30
15,188,95.65,13.95
16,174,97.66,14.70


## 8. Achados preliminares e próximos passos

**Achados (base simulada, 5.000 pedidos de 01/09 a 31/10/2026):**
- **Status:** 88,9% entregues no prazo, 6,9% com atraso e 4,2% cancelados. OTD geral de 92,8%.
- **Pico de trânsito:** o OTD cai de 96,0% para 79,9% no pico, e o tempo mediano sobe de 14,7 para 18,7 min. Os piores horários são 17h (78,7%) e 8h (85,6%). O almoço e o jantar têm mais pedidos, mas não são os horários de pior desempenho: demanda e trânsito são coisas diferentes.
- **Entre cidades:** 21% das entregas cruzam o limite do município. Elas percorrem em média 10,8 km (contra 3,4 km dentro da cidade) e têm OTD de 84,5% (contra 95,0%). O prazo cresce com a distância, mas a variação do tempo também cresce, e o atraso aparece mais nas viagens longas.
- **Cidades:** São Bernardo do Campo e Santo André somam cerca de 56% das coletas. São Caetano do Sul, a menor cidade, tem as menores distâncias (3,9 km) e o melhor OTD (95,1%). Mauá tem a maior distância média de destino (6,0 km), por ficar mais afastada das demais.
- **Bairros:** os Centros concentram as coletas (o Centro de SBC lidera com 372), e as entregas se espalham pelos bairros residenciais.
- **Qualidade dos dados:** 53 entregas com velocidade média abaixo de 3 km/h. São tempos registrados com erro (máximo de 371,8 min) e devem ser tratados no notebook de limpeza.


**Limitações:**
- Dados simulados: as relações encontradas refletem as regras do gerador, não uma operação real.
- Posição dos bairros é simulada (não geocodificada); distância é linha reta (haversine) x 1,3, não a rota real pela malha viária.
- A rua do endereço é sorteada entre vias reais da cidade; a combinação rua x bairro não foi verificada.

**Próximos passos:**
1. `02_limpeza`: tratar outliers de tempo e registrar a regra em `docs/dicionario_dados.md`.
2. Mover `resumo_operacional` para `src/logistica_rotas/metricas/` com teste em `tests/`.
3. Gráficos: distribuição do tempo de trânsito, OTD por hora e por cidade/bairro de destino.
4. Geocodificar os bairros (coordenadas reais) para mapas e roteirização.